# FINANCE 384 Assignment 1 – Part A

## Task A.3: Richer Prediction Model – Gradient Boosting

This standalone notebook implements **Gradient Boosting Regression** as the single richer model selected for comparison with pooled OLS.

The model is chosen because it can represent **nonlinear predictor effects, threshold effects, and interactions** among stock characteristics without requiring those relationships to be specified manually.

This notebook deliberately does **not** perform final hyperparameter selection. A small reference specification is fitted on the **training sample only** to verify the model pipeline. Formal tuning using the validation sample is deferred to **Task A.4**.


### Files required

Upload these files into the Colab/Jupyter working directory before running:

- `FINANCE384_assignmentA_development_panel.csv`
- `FINANCE384_stock_month_data_dictionary.csv`

`FINANCE384_market.csv` is not required for A.3.


In [1]:
# A.3.1 Imports and file paths

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

RANDOM_STATE = 384


In [2]:
# A.3.2 Load the supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Unique stocks:", panel["permno"].nunique())
print("Unique months:", panel["date"].dt.to_period("M").nunique())
print("Duplicate stock-month rows:", panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Unique months: 396
Duplicate stock-month rows: 0


### Base predictor information

The richer model uses the **same base predictor information** as the OLS benchmark:

- 18 admissible numeric stock/market characteristics;
- Fama–French 49 industry membership (`ff49_code`).

Identifiers and realised outcome variables are excluded.


In [3]:
# A.3.3 Define predictors explicitly

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [x for x in numeric_predictors if x != "down_market"]
binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

feature_columns = (
    continuous_predictors
    + binary_predictors
    + categorical_predictors
)

print("Numeric predictors:", len(numeric_predictors))
print("Categorical predictor:", categorical_predictors)
print("Total raw feature columns:", len(feature_columns))


Numeric predictors: 18
Categorical predictor: ['ff49_code']
Total raw feature columns: 19


### Construct the next-month target

The target is \(r^e_{i,t+1}\), constructed from `ret_excess_t`.

A target is retained only if the following observation for the same stock occurs in the **immediately following calendar month**. This prevents a stock that leaves and later re-enters the panel from being incorrectly treated as having a one-month-ahead return across a gap.


In [4]:
# A.3.4 Construct the next-month excess-return target

analysis = panel.sort_values(["permno", "date"]).copy()

analysis["next_date"] = analysis.groupby("permno")["date"].shift(-1)
analysis["ret_excess_t1"] = analysis.groupby("permno")["ret_excess_t"].shift(-1)

analysis["is_consecutive_next_month"] = (
    analysis["next_date"].dt.to_period("M")
    == analysis["date"].dt.to_period("M") + 1
)

analysis.loc[
    ~analysis["is_consecutive_next_month"],
    "ret_excess_t1"
] = np.nan

gapped_observations = (
    analysis["next_date"].notna()
    & ~analysis["is_consecutive_next_month"]
).sum()

analysis_valid = analysis.loc[
    analysis["ret_excess_t1"].notna()
].copy()

print("Raw stock-month rows:", len(analysis))
print("Non-consecutive next observations detected:", int(gapped_observations))
print("Rows with valid next-month targets:", len(analysis_valid))


Raw stock-month rows: 198298
Non-consecutive next observations detected: 30
Rows with valid next-month targets: 197016


### Fixed chronological samples

Gradient Boosting follows a different estimation protocol from OLS:

- **Training:** January 1990–December 2014 — fit candidate models.
- **Validation:** January 2015–December 2018 — used only in A.4 to choose hyperparameters.
- **Test:** January 2019–November 2022 — remains untouched until final evaluation.

This A.3 notebook fits only a **reference model on the training sample**. It does not select final hyperparameters.


In [5]:
# A.3.5 Apply the required chronological split

train = analysis_valid.loc[
    (analysis_valid["date"] >= "1990-01-01")
    & (analysis_valid["date"] <= "2014-12-31")
].copy()

validation = analysis_valid.loc[
    (analysis_valid["date"] >= "2015-01-01")
    & (analysis_valid["date"] <= "2018-12-31")
].copy()

test = analysis_valid.loc[
    (analysis_valid["date"] >= "2019-01-01")
    & (analysis_valid["date"] <= "2022-11-30")
].copy()

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Decision period": [
        "Jan 1990-Dec 2014",
        "Jan 2015-Dec 2018",
        "Jan 2019-Nov 2022",
    ],
    "Months": [
        train["date"].dt.to_period("M").nunique(),
        validation["date"].dt.to_period("M").nunique(),
        test["date"].dt.to_period("M").nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

split_summary


,Sample,Decision period,Months,Stock-month rows
0,Training,Jan 1990-Dec 2014,300,149334
1,Validation,Jan 2015-Dec 2018,48,24051
2,Test,Jan 2019-Nov 2022,47,23631


### Preprocessing

The same base feature information is retained across OLS and Gradient Boosting.

For the richer model:

- continuous predictors are median-imputed and standardised using **training-sample parameters**;
- `down_market` remains binary;
- `ff49_code` is one-hot encoded;
- no validation or test information is used to estimate preprocessing parameters.

Standardisation is not required mathematically for tree splits, but retaining the same feature-processing framework keeps the modelling pipeline consistent across the assessed comparison.


In [6]:
# A.3.6 Define and fit the preprocessing transformation

continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_predictors),
        ("binary", binary_pipeline, binary_predictors),
        ("industry", categorical_pipeline, categorical_predictors),
    ],
    remainder="drop",
)

X_train_raw = train[feature_columns]
y_train = train["ret_excess_t1"]

X_validation_raw = validation[feature_columns]
y_validation = validation["ret_excess_t1"]

X_test_raw = test[feature_columns]
y_test = test["ret_excess_t1"]

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("Test matrix:", X_test.shape)


Training matrix: (149334, 64)
Validation matrix: (24051, 64)
Test matrix: (23631, 64)


### Reference Gradient Boosting specification

The following model is fitted only as a **pipeline check** for A.3. These settings are **not the final selected hyperparameters**:

- `n_estimators = 50`
- `learning_rate = 0.05`
- `max_depth = 2`
- squared-error loss

A.4 will compare a reproducible hyperparameter grid using pooled validation RMSE and select the final training-fitted model.


In [7]:
# A.3.7 Define and fit a reference Gradient Boosting model on TRAINING ONLY

gb_reference = GradientBoostingRegressor(
    loss="squared_error",
    n_estimators=50,
    learning_rate=0.05,
    max_depth=2,
    random_state=RANDOM_STATE
)

gb_reference.fit(X_train, y_train)

print("Reference Gradient Boosting model fitted successfully.")
print("Training observations:", X_train.shape[0])
print("Number of transformed predictors:", X_train.shape[1])
print("Reference n_estimators:", gb_reference.n_estimators)
print("Reference learning_rate:", gb_reference.learning_rate)
print("Reference max_depth:", gb_reference.max_depth)


Reference Gradient Boosting model fitted successfully.
Training observations: 149334
Number of transformed predictors: 64
Reference n_estimators: 50
Reference learning_rate: 0.05
Reference max_depth: 2


### Sanity-check predictions

The next cell verifies that the fitted reference model can generate validation predictions with no missing values.

**These validation predictions are not used to choose the final model in A.3.** Formal comparison of candidate hyperparameters is deferred to A.4.


In [8]:
# A.3.8 Generate reference validation predictions for pipeline checking only

gb_reference_validation_pred = gb_reference.predict(X_validation)

reference_validation_predictions = validation[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

reference_validation_predictions = reference_validation_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)

reference_validation_predictions[
    "gb_reference_pred_excess_return_t1"
] = gb_reference_validation_pred

print("Reference validation predictions:", len(reference_validation_predictions))
print(
    "Missing predictions:",
    int(
        reference_validation_predictions[
            "gb_reference_pred_excess_return_t1"
        ].isna().sum()
    )
)

reference_validation_predictions.head()


Reference validation predictions: 24051
Missing predictions: 0


,date,permno,ticker,actual_excess_return_t1,gb_reference_pred_excess_return_t1
539,2015-01-30,10104,ORCL,0.046073,0.007373
540,2015-02-27,10104,ORCL,-0.015290,0.007373
541,2015-03-31,10104,ORCL,0.014368,0.007373
542,2015-04-30,10104,ORCL,-0.002980,0.007373
543,2015-05-29,10104,ORCL,-0.073350,0.007373


### Feature-importance audit

Gradient Boosting can provide impurity-based feature-importance scores. These are included only as an exploratory model-audit output at A.3 and should not be interpreted as causal effects.

Formal model interpretation belongs to A.7 after the final tuned model has been selected and evaluated.


In [9]:
# A.3.9 Reference-model feature importance

feature_names = preprocessor.get_feature_names_out()

reference_feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": gb_reference.feature_importances_,
})

reference_feature_importance = (
    reference_feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

reference_feature_importance.head(15)


,feature,importance
0,continuous__mkt_12m,0.620208
1,continuous__mkt_vol_12m,0.192735
2,continuous__mom12_2,0.104735
3,continuous__size,0.028266
4,continuous__bm,0.022750
5,continuous__beta60,0.015277
6,continuous__ivol60,0.005208
7,continuous__dollar_volume,0.004909
8,continuous__divyield,0.004688
9,continuous__gross_profit,0.001225


In [10]:
# A.3.10 Final A.3 audit checks

a3_audit = pd.DataFrame({
    "Item": [
        "Chosen richer model",
        "Training observations",
        "Validation observations",
        "Test observations",
        "Transformed predictors",
        "Reference validation predictions",
        "Missing reference validation predictions",
        "Final hyperparameters selected?",
    ],
    "Value": [
        "Gradient Boosting Regression",
        len(train),
        len(validation),
        len(test),
        X_train.shape[1],
        len(reference_validation_predictions),
        int(
            reference_validation_predictions[
                "gb_reference_pred_excess_return_t1"
            ].isna().sum()
        ),
        "No - deferred to A.4",
    ],
})

a3_audit


,Item,Value
0,Chosen richer model,Gradient Boosting Regression
1,Training observations,149334
2,Validation observations,24051
3,Test observations,23631
4,Transformed predictors,64
5,Reference validation predictions,24051
6,Missing reference validation predictions,0
7,Final hyperparameters selected?,No - deferred to A.4


## A.3 Summary

Gradient Boosting Regression is established as the single richer model for the assessed comparison with pooled OLS. It uses the same base stock and market information while allowing nonlinearities, threshold effects, and interactions among predictors.

A reference specification has been fitted successfully on the **training sample only** to confirm that the modelling pipeline works. The validation and test samples have not been used to estimate model parameters.

The reference hyperparameters are **not final**. Task A.4 will conduct the required reproducible hyperparameter search using the **validation sample only**, with pooled validation RMSE as the primary selection criterion. The selected training-sample fitted model will then be retained without re-estimation on training + validation.
